# Пример · #901 dropout (эталон) · **easy**

Сначала сам: [901_dropout_torch.ipynb](./901_dropout_torch.ipynb) · конспект [d.4.37 §2.2](../../../d.4.37%20Регуляризация%20в%20DL.md#dropout--случайно-выключаем-нейроны-на-обучении) · PyTorch [15_dropout](../../../../../../pytorch/pytorch%20дубинин/15/15_dropout.ipynb).

Здесь — **готовый код** + разбор **каждой строки**: что делает inverted dropout и зачем `training`/`eval`.

## Зачем эта задача

В сети Dropout — не «магия PyTorch», а три правила:

1. На **обучении** случайно гасим часть значений (нейронов).
2. Чтобы сумма сигнала не **рухнула вдвое** (при `p=0.5` выживает ~половина), выжившие **масштабируем** — это **inverted dropout** (так же в `nn.Dropout`).
3. На **инференсе** (`eval`) dropout **выключен**: все нейроны работают, вход = выход.

Задача #901 проверяет, что ты умеешь повторить эту логику на тензорах **без** `nn.Dropout` — тогда `model.train()` / `model.eval()` и [видео 15](https://www.youtube.com/watch?v=42xf6g4yPIA) становятся понятными.

## Условие (как в Deep-ML)

Написать `dropout(x, p, training)`:

- `p` — вероятность **обнулить** элемент (как в `nn.Dropout(p)`).
- `training=False` → вернуть `x` без изменений.
- `training=True` → **inverted** dropout: маска + деление на `(1 - p)`.

Проверки:

```python
x = torch.ones(1000)
y = dropout(x, 0.5, training=True)
assert torch.allclose(x.mean(), y.mean(), atol=0.15)  # среднее ≈ как у входа
assert torch.allclose(dropout(x, 0.5, training=False), x)
```

In [ ]:
import torch


def dropout(x, p, training):
    if not training:
        return x
    mask = (torch.rand_like(x) > p).float()
    return x * mask / (1.0 - p)


x = torch.ones(1000)
y = dropout(x, 0.5, training=True)
assert torch.allclose(x.mean(), y.mean(), atol=0.15)
assert torch.allclose(dropout(x, 0.5, training=False), x)
print("ok")

## Построчно: функция `dropout`

### `import torch`

Нужны тензоры, `rand_like` (случайные числа той же формы, что `x`) и сравнения поэлементно.

### `if not training: return x`

**Зачем:** в реальной сети это роль `model.eval()`. На тесте мы **не** хотим случайно выкидывать нейроны — иначе ответ на одной и той же картинке каждый раз разный.

**Почему без масштаба:** на train мы учили «ансамбль дырявых подсетей», на test используем **полную** сеть. Именно поэтому на train включается деление на `(1-p)` — чтобы **ожидаемая** величина активации совпадала с eval, когда все нейроны включены (см. [d.4.37 §2.2](../../../d.4.37%20Регуляризация%20в%20DL.md)).

### `mask = (torch.rand_like(x) > p).float()`

| Кусок | Смысл |
|---|---|
| `torch.rand_like(x)` | Для **каждого** элемента `x` — своё число Uniform(0, 1). Форма как у `x` (тут 1000 единиц → 1000 случайностей). |
| `> p` | Порог: при `p=0.5` примерно половина чисел больше 0.5 → `True`, половина → `False`. Это и есть «выкинуть с вероятностью `p`»: **выживаем**, если `rand > p`. |
| `.float()` | Булеву маску превращаем в 0.0 / 1.0, чтобы умножать на `x` (как в `nn.Dropout`). |

**Полезно для обучения:** каждый forward на train — **новая** маска → сеть не может зазубрить одного «любимого» нейрона (регуляризация).

**Связь с Бернулли:** элемент остаётся с вероятностью `(1 - p)` — как в описании [#151](../151_dropout_layer%20—%20medium/) (NumPy-версия).

### `return x * mask / (1.0 - p)`

1. **`x * mask`** — где маска 0, значение обнулили (нейрон «отдыхает»); где 1 — оставили.
2. **`/ (1.0 - p)`** — **inverted dropout**. При `p=0.5` делим на `0.5`, то есть выжившие умножаем на **2**.

**Математика ожидания** (почему это полезно):

Пусть один элемент входа = 1. С вероятностью `(1-p)` он выжил и стал `1/(1-p)`; с вероятностью `p` стал 0.

$$\mathbb{E}[\text{выход}] = (1-p)\cdot\frac{1}{1-p} + p\cdot 0 = 1$$

То есть **в среднем** после dropout сигнал того же масштаба, что вход — как если бы на eval все нейроны были включены. Без деления на `(1-p)` среднее было бы ~0.5 при `p=0.5` и следующий слой получал бы «тихий» сигнал только на train.

**Граничный случай:** при `p → 1` делить на `(1-p)` нельзя — в задаче и в PyTorch обычно `p < 1`.

## Построчно: проверки

### `x = torch.ones(1000)`

Игрушечный вход: 1000 одинаковых единиц. Удобно смотреть **среднее**: у входа `mean = 1.0`.

### `y = dropout(x, 0.5, training=True)`

Один случайный прогон с `p=0.5`. Маска каждый раз другая — поэтому `y` не воспроизводится бит в бит, но **статистика** должна сходиться.

### `assert torch.allclose(x.mean(), y.mean(), atol=0.15)`

- Проверяем **inverted**: среднее выхода **близко** к 1, а не ~0.5.
- `atol=0.15` — допуск из-за **конечной** выборки (1000 чисел, а не бесконечность). На 1000 элементах разброс случайный; 0.15 — щедрый запас, чтобы assert не падал от везения/невезения.
- Если убрать деление на `(1-p)`, типичное среднее было бы около 0.5 — assert бы ломался.

### `assert torch.allclose(dropout(x, 0.5, training=False), x)`

Режим **инференса**: тензор не меняется (ни маски, ни масштаба). Так и ведёт себя слой внутри `model.eval()`.

In [ ]:
# Наглядно: с масштабом vs без (одна маска, p=0.5)
torch.manual_seed(0)
x_small = torch.ones(8)
mask = (torch.rand_like(x_small) > 0.5).float()
plain = x_small * mask
inverted = plain / 0.5
print("mask:", mask.tolist())
print("без /(1-p), mean =", plain.mean().item())
print("inverted,  mean =", inverted.mean().item())

## Связь с PyTorch

- Твоя функция ≈ то, что делает `nn.Dropout(p)` при `training=True` (inverted dropout по умолчанию).
- Флаг `training` в задаче = то, что задаётся через `module.train()` / `module.eval()` на всей модели.
- После этой задачи логично: [901 задача](https://www.deep-ml.com/problems/901) на сайте → [15_dropout](../../../../../../pytorch/pytorch%20дубинин/15/15_dropout.ipynb) → `FashionMLP` в project (`train`/`eval`).

**Что сказать на собесе:** dropout на train случайно обнуляет + масштабирует выживших; на eval выключен; иначе предсказания «дрожат».